# SACDlive Dox phase diagram: Hoechst–SPEN

This pipeline processes the complete July 1 ONI dual-view, single-time-point z-stack dataset. Each 25-frame movie is split into left Hoechst and right SPEN channels, reconstructed independently with SACDpy, assembled into channel-specific z-stacks and z-MIPs, segmented with CellposeSAM, and exported as padded per-nucleus review files.

Each retained nucleus receives one three-channel TIFF (Hoechst SACD, SPEN SACD, binary mask) and one annotated SPEN PNG. Quantitative SPEN measurements use a core eroded inward by one third of the mask's area-equivalent radius to reduce neighboring-cell spillover. Raw mean-frame MIPs remain only in the FOV reconstruction archive.

## 1. Imports and environment

Run this notebook with the `conda-smlm` kernel from the SACDpy repository folder.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import tifffile

repo = Path.cwd()
src = repo / 'src'
if src.exists() and str(src) not in sys.path:
    sys.path.insert(0, str(src))

from sacdpy.phase_diagram import (
    DEFAULT_DATASET_ROOT, DEFAULT_OUTPUT_ROOT, PhaseDiagramConfig,
    discover_phase_fovs, repackage_existing_nuclei, run_phase_dataset,
)

print(f'Working folder: {repo}')
print(f'Python: {sys.executable}')

## 2. Configuration

`run_mode='pilot'` processes one representative FOV per biological condition. Set it to `'full'` for the complete resumable 82-FOV run. Existing FOVs with validated `_COMPLETE.json` records are skipped unless `overwrite=True`.

In [ ]:
dataset_root = DEFAULT_DATASET_ROOT
output_root = DEFAULT_OUTPUT_ROOT
run_mode = 'full'  # 'pilot' or 'full'
overwrite = False
repackage_only = True  # Rebuild nucleus review files from completed outputs; no SACD or Cellpose rerun

config = PhaseDiagramConfig(
    dataset_root=dataset_root,
    output_root=output_root,
    hoechst_wavelength_nm=461.0,
    spen_wavelength_nm=647.0,
    mag=2, iter1=7, iter2=8, ac_order=2, subfactor=0.8,
    cellpose_diameter_px=140.0,
    cellpose_flow_threshold=0.4,
    cellpose_cellprob_threshold=0.0,
    crop_padding_px=10,
    core_erosion_radius_fraction=1/3,
    overwrite=overwrite,
)
config

## 3. Read-only dataset preflight

Confirm FOV, condition, z-plane, frame, and dual-view dimensions before processing.

In [ ]:
plans = discover_phase_fovs(dataset_root)
condition_counts = {}
for plan in plans:
    condition_counts[plan.condition] = condition_counts.get(plan.condition, 0) + 1

total_movies = sum(len(plan.files) for plan in plans)
print(f'FOVs: {len(plans)} | z-plane movies: {total_movies}')
print('Conditions:', condition_counts)
for plan in plans:
    first = tifffile.imread(plan.files[plan.z_indices[0]])
    if first.ndim != 3 or first.shape[-1] % 2:
        raise ValueError(f'Unexpected dual-view input shape for {plan.fov_name}: {first.shape}')
    print(f'{plan.fov_name}: Z={len(plan.z_indices)}, movie={first.shape}, per-channel={first.shape[:-1] + (first.shape[-1] // 2,)}')

if len(plans) != 82 or total_movies != 581:
    raise AssertionError(f'Expected 82 FOVs and 581 z-plane movies, found {len(plans)} and {total_movies}')

## 4. Choose representative pilots

The first FOV from each biological condition is used when `run_mode='pilot'`. Pilot outputs are written to their final locations and will be skipped during the subsequent full run.

In [ ]:
pilot_names = []
seen_conditions = set()
for plan in plans:
    if plan.condition not in seen_conditions:
        pilot_names.append(plan.fov_name)
        seen_conditions.add(plan.condition)
print('Pilot FOVs:')
for name in pilot_names:
    print(' -', name)

## 5. Run or repackage the dataset

With `repackage_only=True`, this reads the completed SACD MIPs and Cellpose masks and replaces only the nucleus review products. Set it to `False` for a new resumable SACD/Cellpose run.

In [ ]:
if repackage_only:
    completions, failures = repackage_existing_nuclei(config, expected_nuclei=1880)
else:
    selected_fovs = pilot_names if run_mode == 'pilot' else None
    completions, failures = run_phase_dataset(
        config,
        fov_names=selected_fovs,
        stop_on_error=False,
    )
print(f'Completed: {len(completions)} | failed: {len(failures)}')
if failures:
    for failure in failures:
        print(failure)

## 6. Review summary and first segmentation overlay

In [ ]:
summary_path = output_root / 'manifests' / 'run_summary.json'
summary = json.loads(summary_path.read_text())
display(summary)

overlays = sorted((output_root / 'qc').rglob('*__CellposeSAM-overlay.png'))
if overlays:
    image = plt.imread(overlays[0])
    plt.figure(figsize=(8, 12))
    plt.imshow(image)
    plt.title(overlays[0].name)
    plt.axis('off')
    plt.show()

## 7. Manual review and rough phase-diagram handoff

Review the core-intensity-ranked TIFF/PNG pairs in `nuclei/dSPEN_FL` and `nuclei/dSPEN_dRRM`. The solid contour is the CellposeSAM boundary and the dotted contour is the eroded measurement core. Edit `manifests/nucleus_manifest.csv` to set `manual_use` and `manual_note`; automatic exclusions are listed separately in `manifests/excluded_nuclei.csv`. The three-score comparison is saved as `qc/phase_diagram-SPEN_core_mean-vs-dispersion.png`.